<a href="https://colab.research.google.com/github/Saiful-2/forest-cover-classification/blob/main/notebooks/10_streamlit_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
import numpy as np
import pandas as pd
import joblib

# -----------------------------
# Load Model
# -----------------------------
model = joblib.load("model.pkl")

st.set_page_config(page_title="Forest Cover Predictor", layout="centered")

st.title("🌲 Forest Cover Type Prediction App")
st.write("Predict forest cover type using environmental features")

# -----------------------------
# INPUT SECTION
# -----------------------------
st.header("📥 Input Features")

col1, col2 = st.columns(2)

with col1:
    Elevation = st.number_input("Elevation", 0, 5000, 2500)
    Aspect = st.number_input("Aspect", 0, 360, 100)
    Slope = st.number_input("Slope", 0, 90, 10)
    Horizontal_Distance_To_Hydrology = st.number_input("Dist to Hydrology (Horizontal)", 0, 10000, 100)
    Vertical_Distance_To_Hydrology = st.number_input("Dist to Hydrology (Vertical)", -500, 500, 0)

with col2:
    Horizontal_Distance_To_Roadways = st.number_input("Dist to Roadways", 0, 10000, 1000)
    Hillshade_9am = st.number_input("Hillshade 9am", 0, 255, 200)
    Hillshade_Noon = st.number_input("Hillshade Noon", 0, 255, 220)
    Hillshade_3pm = st.number_input("Hillshade 3pm", 0, 255, 180)
    Horizontal_Distance_To_Fire_Points = st.number_input("Dist to Fire Points", 0, 10000, 500)

# -----------------------------
# Wilderness Area
# -----------------------------
st.subheader("🌄 Wilderness Area")

wa1 = st.checkbox("Rawah")
wa2 = st.checkbox("Neota")
wa3 = st.checkbox("Comanche Peak")
wa4 = st.checkbox("Cache la Poudre")

# -----------------------------
# Soil Type
# -----------------------------
st.subheader("🌱 Soil Type")

soil_type = st.selectbox("Select Soil Type (0–39)", list(range(40)))

soil_features = [0]*40
soil_features[soil_type] = 1

Soil_Sum = sum(soil_features)

# -----------------------------
# PREDICTION
# -----------------------------
if st.button("🔮 Predict Forest Cover Type"):

    input_data = [
        Elevation, Aspect, Slope,
        Horizontal_Distance_To_Hydrology,
        Vertical_Distance_To_Hydrology,
        Horizontal_Distance_To_Roadways,
        Hillshade_9am, Hillshade_Noon, Hillshade_3pm,
        Horizontal_Distance_To_Fire_Points,
        int(wa1), int(wa2), int(wa3), int(wa4),
        Soil_Sum
    ] + soil_features

    st.write("Total features:", len(input_data))

    columns = [
        'Elevation', 'Aspect', 'Slope',
        'Horizontal_Distance_To_Hydrology',
        'Vertical_Distance_To_Hydrology',
        'Horizontal_Distance_To_Roadways',
        'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm',
        'Horizontal_Distance_To_Fire_Points',
        'Wilderness_Area1',
        'Wilderness_Area2',
        'Wilderness_Area3',
        'Wilderness_Area4',
        'Soil_Sum'
    ] + [f"Soil_Type{i}" for i in range(1, 41)]

    input_df = pd.DataFrame([input_data], columns=columns)

    prediction = model.predict(input_df)

    cover_types = {
        1: "Spruce/Fir",
        2: "Lodgepole Pine",
        3: "Ponderosa Pine",
        4: "Cottonwood/Willow",
        5: "Aspen",
        6: "Douglas-fir",
        7: "Krummholz"
    }

    result = cover_types.get(prediction[0], "Unknown")

    st.success(f"🌲 Predicted Cover Type: {result}")